# RSNA Knee OrthoFoundation full

New frozen DINOv3-L knee encoder, normalized CLS per MRI slice, freshly initialized 12-target study head. Primary forward follows the author's no-RoPE wrapper; this is explicitly recorded. The pilot verifies strict weights, finite features, one finite head update, throughput and memory. Full mode stops if measured extraction exceeds its budget. Gold studies are excluded from gradients; fixed final epochs are used without Gold checkpoint selection. No leaderboard gain is assumed.

In [ ]:
print('{"event":"bootstrap_start","diagnostic_schema":1}',flush=True)
import json,os,time,traceback
BOOTSTRAP_DIR='/kaggle/working'
BOOTSTRAP_STAGE='notebook_start'
os.makedirs(BOOTSTRAP_DIR,exist_ok=True)
def bootstrap_checkpoint(stage):
    global BOOTSTRAP_STAGE
    BOOTSTRAP_STAGE=stage
    event={'diagnostic_schema':1,'event':'bootstrap_checkpoint','stage':stage,'unix_seconds':time.time()}
    with open(os.path.join(BOOTSTRAP_DIR,'bootstrap_progress.json'),'w') as stream:
        json.dump(event,stream,indent=2)
    with open(os.path.join(BOOTSTRAP_DIR,'bootstrap_events.jsonl'),'a') as stream:
        stream.write(json.dumps(event)+'\n')
    print(json.dumps(event),flush=True)
def bootstrap_save_failure(error):
    event={'diagnostic_schema':1,'event':'bootstrap_failure','stage':BOOTSTRAP_STAGE,
           'error_type':type(error).__name__,'error':str(error),'traceback':traceback.format_exc(),
           'unix_seconds':time.time()}
    try:
        with open(os.path.join(BOOTSTRAP_DIR,'bootstrap_failure.json'),'w') as stream:
            json.dump(event,stream,indent=2)
    except BaseException as save_error:
        print(json.dumps({'event':'bootstrap_failure_save_error','error_type':type(save_error).__name__}),flush=True)
    print(json.dumps(event),flush=True)
bootstrap_checkpoint('notebook_start')
with open(os.path.join(BOOTSTRAP_DIR,'bootstrap_start.json'),'w') as stream:
    json.dump({'diagnostic_schema':1,'event':'bootstrap_start','unix_seconds':time.time()},stream,indent=2)


In [ ]:
COMPRESSED_PAYLOAD='...'
SETTINGS={'stage': 'full', 'config': {'experiment': 'orthofoundation_frozen_attention_v1', 'encoder': 'dinov3_vitl16', 'extraction_variant': 'author_no_rope', 'feature_dim': 1024, 'checkpoint_sha256': '385a775822107b68eaa486336feb982e1ce7bd6d4e8c03ceb482a0bf546f2ff9', 'checkpoint_bytes': 1213056638, 'resolution': 224, 'slices_per_slot': 4, 'fov': 0.92, 'image_batch_size': 16, 'workers': 2, 'pilot_studies': 16, 'max_extract_seconds': 9000, 'head_kind': 'attention', 'head_dim': 256, 'head_dropout': 0.2, 'head_epochs': 12, 'head_batch_size': 64, 'head_lr': 0.001, 'head_weight_decay': 0.01, 'weak_holdout_fold': 0, 'seed': 42}, 'source_sha256': {'orthofoundation/__init__.py': 'f3df43207f5d18497b4f698aa91c746bf81dcd4edb74b81784136acd8aa52cf1', 'orthofoundation/cache.py': '700767576eb7b6d9dfa22f77bdf2088411ba4bdf6036c80db58bf608c96f7485', 'orthofoundation/data.py': 'f3b63ab922f0452c065126dc2ef1c87442a0b988dff29d25c201666a8a5310aa', 'orthofoundation/encoder.py': '86defa5f943bfc161603c7b377bbf08a9d5c2a270d01b1df5c039ab59cff7ecb', 'orthofoundation/heads.py': 'c814b1bcbf1ac1322b717c1f1e6ba7b9173357e956e9104b2c77d0351a833f1f', 'orthofoundation/run.py': '740c2e2599516b1876e7d7226bb0f11c51205cbdaec0b6eebf44dc740c659e10', 'orthofoundation/train.py': '928121a030aabf4fc58594da967ad98ff43a7ce915474d69a909ab09652a7dc1'}, 'cache_kernel_ids': ['alanchoo/rsna-knee-cnxt-cache-0-zip', 'alanchoo/rsna-knee-cnxt-cache-1-zip', 'alanchoo/rsna-knee-cnxt-cache-2-zip', 'alanchoo/rsna-knee-cnxt-cache-3-zip', 'alanchoo/rsna-knee-cnxt-cache-4-zip', 'alanchoo/rsna-knee-cnxt-cache-5-direct-zip', 'alanchoo/rsna-knee-cnxt-cache-6-zip'], 'assets_dataset': 'alanchoo/orthofoundation-l-assets-v1', 'training_inputs_dataset': 'alanchoo/orthofoundation-l-assets-v1', 'preprocess_sha256': '68601ad84ec3a0f030b5c2d37cc0d657cd3e12b17f7130e7385fba9648d324b6', 'resume_kernel': 'alanchoo/rsna-knee-orthofoundation-pilot'}
try:
    bootstrap_checkpoint('imports')
    from pathlib import Path
    import base64,hashlib,json,os,subprocess,sys,tarfile,zlib
    bootstrap_checkpoint('vendoring')
    CODE=Path('/kaggle/working/source'); CODE.mkdir(exist_ok=True)
    for name,encoded in json.loads(zlib.decompress(base64.b64decode(COMPRESSED_PAYLOAD))).items():
        target=CODE/name; target.parent.mkdir(exist_ok=True); target.write_bytes(base64.b64decode(encoded))
        if hashlib.sha256(target.read_bytes()).hexdigest()!=SETTINGS['source_sha256'][name]:
            raise RuntimeError('Vendored source SHA mismatch: '+name)
    (CODE/'config.json').write_text(json.dumps(SETTINGS['config'],indent=2))
    bootstrap_checkpoint('input_resolution')
    INPUT=Path('/kaggle/input')
    def find_named_input(identifier,marker,kind='datasets'):
        owner,slug=identifier.split('/')
        candidates=[INPUT/kind/owner/slug,INPUT/slug,INPUT/'kernels'/owner/slug,INPUT/owner/slug]
        matches=[path for path in candidates if (path/marker).is_file()]
        if not matches:
            for pattern in [f'*/{slug}',f'*/*/{slug}',f'*/*/*/{slug}']:
                matches.extend(path for path in INPUT.glob(pattern) if (path/marker).is_file())
        matches=list(dict.fromkeys(matches))
        if len(matches)!=1:raise RuntimeError(f'Expected one {identifier} source with {marker}: {matches}')
        return matches[0]
    competition=[path for path in [INPUT/'competitions/rsna-knee-abnormality-detection',INPUT/'rsna-knee-abnormality-detection']
                 if (path/'train.csv').is_file() and (path/'train_series.csv').is_file()]
    if len(competition)!=1:raise RuntimeError('Missing/ambiguous competition source')
    assets=find_named_input(SETTINGS['assets_dataset'],'asset_manifest.json')
    training=find_named_input(SETTINGS['training_inputs_dataset'],'labels_v0.csv')
    manifest=json.loads((assets/'asset_manifest.json').read_text())
    expected=SETTINGS['config']
    if manifest['checkpoint']['sha256']!=expected['checkpoint_sha256'] or manifest['checkpoint']['bytes']!=expected['checkpoint_bytes']:
        raise RuntimeError('Asset manifest checkpoint differs from frozen experiment')
    bootstrap_checkpoint('vendor_unpack')
    archive=assets/manifest['dinov3_source']['filename']
    if hashlib.sha256(archive.read_bytes()).hexdigest()!=manifest['dinov3_source']['sha256']:
        raise RuntimeError('DINOv3 source archive SHA mismatch')
    with tarfile.open(archive,'r:gz') as tar:
        # Reject links and traversal even before Python's extraction filter.
        for member in tar.getmembers():
            parts=Path(member.name).parts
            if member.issym() or member.islnk() or Path(member.name).is_absolute() or '..' in parts:
                raise RuntimeError('Unsafe DINOv3 source archive member')
        tar.extractall(CODE,filter='data')
    bootstrap_checkpoint('cache_resolution')
    roots=[find_named_input(identifier,'cache_summary.json','notebooks') for identifier in SETTINGS['cache_kernel_ids']]
    bootstrap_checkpoint('gpu_selection')
    env=dict(os.environ); env['PYTHONPATH']=str(CODE)+os.pathsep+env.get('PYTHONPATH','')
    env['OMP_NUM_THREADS']='1'; env['OPENBLAS_NUM_THREADS']='1'
    import torch
    if not torch.cuda.is_available():raise RuntimeError('No CUDA GPU; refuse CPU fallback')
    free=[torch.cuda.mem_get_info(index)[0] for index in range(torch.cuda.device_count())]
    env['CUDA_VISIBLE_DEVICES']=str(max(range(len(free)),key=lambda index:free[index]))
    cmd=[sys.executable,'-m','orthofoundation.run','--stage',SETTINGS['stage'],'--config',str(CODE/'config.json'),
         '--data-dir',str(competition[0]),'--labels',str(training/'labels_v0.csv'),
         '--cache-roots',*[str(path) for path in roots],'--cache-fingerprints',str(training/'cache_input_fingerprints.json'),
         '--preprocess-sha256',SETTINGS['preprocess_sha256'],'--dinov3-repo',str(CODE/manifest['dinov3_source']['archive_root']),
         '--checkpoint',str(assets/manifest['checkpoint']['filename']),'--assets-manifest',str(assets/'asset_manifest.json'),
         '--output-dir','/kaggle/working/run']
    if SETTINGS['resume_kernel']:
        previous=find_named_input(SETTINGS['resume_kernel'],'run/provenance.json','notebooks')
        cmd+=['--resume-features',str(previous/'run/features')]
    print(json.dumps({'launch':cmd,'gpu_free_gb':[value/1e9 for value in free]}),flush=True)
    bootstrap_checkpoint('launch')
    subprocess.run(cmd,env=env,check=True)
    bootstrap_checkpoint('report')
    report_file=Path('/kaggle/working/run')/('pilot_report.json' if SETTINGS['stage']=='pilot' else 'training_report.json')
    report=json.loads(report_file.read_text())
    if SETTINGS['stage']=='full' and not report['training_complete']:raise RuntimeError('Full training did not finish')
    print(json.dumps(report,indent=2),flush=True)
    bootstrap_checkpoint('complete')
except BaseException as bootstrap_error:
    bootstrap_save_failure(bootstrap_error)
    raise
